In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # 0 = all messages, 1 = filter INFO, 2 = filter WARNING, 3 = filter ERROR
import tensorflow as tf
# … rest of your imports and code …

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- 1) Định nghĩa các trọng số (theo đúng kích thước bạn đã cho) ---
# Wx: (hidden_size=3, input_size=4)
Wx = tf.constant([[ 1,  0, -1,  1],
                  [ 0,  2,  1,  0],
                  [ 1,  2,  1,  0]], dtype=tf.float32)
# Wh: (hidden_size=3, hidden_size=3)
Wh = tf.constant([[1, 0, 1],
                  [0, 1, 1],
                  [0, 1, 0]], dtype=tf.float32)
# bh: (hidden_size=3,)
bh = tf.constant([1, -1, -1], dtype=tf.float32)

# Wy: (output_size=1, hidden_size=3)
Wy = tf.constant([[ 3, 0, -1]], dtype=tf.float32)
# by: (output_size=1,)
by = tf.constant([3.], dtype=tf.float32)


# --- 2) Xây dựng model Keras ---
#   - SimpleRNN: units=3, activation='tanh', return_sequences=True để lấy H[t] cho mỗi bước
#   - Dense: 1 neuron, activation='sigmoid' cho đầu ra Y[t]

rnn = layers.SimpleRNN(
    units=3,
    activation='tanh',
    return_sequences=True,
    # kernel: shape (input_dim, units)  = (4,3)  ← Wx^T
    kernel_initializer=keras.initializers.Constant(tf.transpose(Wx)),
    # recurrent_kernel: shape (units, units) = (3,3) ← Wh^T
    recurrent_initializer=keras.initializers.Constant(tf.transpose(Wh)),
    # bias: (units,) = bh
    bias_initializer=keras.initializers.Constant(bh),
)

dense = layers.Dense(
    units=1,
    activation='sigmoid',
    # kernel: shape (units_in, units_out) = (3,1) ← Wy^T
    kernel_initializer=keras.initializers.Constant(tf.transpose(Wy)),
    bias_initializer=keras.initializers.Constant(by),
)

model = keras.Sequential([rnn, dense])
# --- 3) Chuẩn bị dữ liệu vào và chạy forward ---
X_seq = [
    [2, -1,  0, 1],
    [1,  0,  1, 0],
    [1,  1,  0, 0],
    [1,  2, -1, 2],
]
# thêm batch dimension: shape = (1, time_steps=4, input_dim=4)
X = tf.constant([X_seq], dtype=tf.float32)

# forward pass
Y = model(X)  # shape = (1, 4, 1)

# flatten và in ra
print("Y[t] = ", tf.reshape(Y, [-1]).numpy())


Y[t] =  [0.99884164 0.9960476  0.9944871  0.993336  ]


In [ ]:
import random
import math
from statistics import mean, variance

# ---- Helper classes ----
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature      # index of feature used for split
        self.threshold = threshold  # split threshold
        self.left = left            # left child node
        self.right = right          # right child node
        self.value = value          # prediction value (for leaf)

    def is_leaf(self):
        return self.value is not None

def mse(values):
    if len(values) == 0:
        return 0.0
    if len(values) == 1:
        return 0.0
    return variance(values)

# ---- Build a decision tree (regression) ----
def build_tree(X, y, depth=0, max_depth=3, min_samples=1, feature_subsample=None):
    # Recursively builds a decision tree.# Prints the process at each step.
    indent = "  " * depth
    # stopping conditions
    if depth >= max_depth or len(set(y)) == 1 or len(X) <= min_samples:
        leaf_value = mean(y)
        print(f"{indent}Leaf node (depth={depth}): predict value = {leaf_value:.3f}")
        return TreeNode(value=leaf_value)

    n_features = len(X[0])
    # Random subset of features (like Random Forest)
    if feature_subsample is None:
        k = int(math.sqrt(n_features)) or 1
    else:
        k = feature_subsample
    features = random.sample(range(n_features), k=k)

    best_feature = None
    best_threshold = None
    best_score = float("inf")
    best_splits = None
    # Search for best split
    for feature in features:
        # get unique thresholds to try
        thresholds = sorted(set([row[feature] for row in X]))
        for t in thresholds:
            left_y = [y[i] for i in range(len(X)) if X[i][feature] <= t]
            right_y = [y[i] for i in range(len(X)) if X[i][feature] > t]
            if len(left_y) == 0 or len(right_y) == 0:
                continue
            score = (len(left_y) * mse(left_y) + len(right_y) * mse(right_y)) / len(y)
            if score < best_score:
                best_score = score
                best_feature = feature
                best_threshold = t
                best_splits = left_y, right_y

    if best_feature is None:
        # Fallback: create leaf
        leaf_value = mean(y)
        print(f"{indent}Leaf node (depth={depth}): predict value = {leaf_value:.3f}")
        return TreeNode(value=leaf_value)

    left_indices = [i for i in range(len(X)) if X[i][best_feature] <= best_threshold]
    right_indices = [i for i in range(len(X)) if X[i][best_feature] > best_threshold]
    X_left = [X[i] for i in left_indices]
    y_left = [y[i] for i in left_indices]
    X_right = [X[i] for i in right_indices]
    y_right = [y[i] for i in right_indices]
    print(f"{indent}Split on feature[{best_feature}] <= {best_threshold:.3f} "
          f"(depth={depth}, impurity={best_score:.3f})")

    left_child = build_tree(X_left, y_left, depth + 1, max_depth, min_samples, feature_subsample)
    right_child = build_tree(X_right, y_right, depth + 1, max_depth, min_samples, feature_subsample)
    return TreeNode(feature=best_feature, threshold=best_threshold,
                    left=left_child, right=right_child)

def predict_tree(node, x):
    while not node.is_leaf():
        if x[node.feature] <= node.threshold:
            node = node.left
        else:
            node = node.right
    return node.value

# ---- Random Forest from scratch ----
class RandomForestRegressorScratch:
    def __init__(self, n_trees=3, max_depth=3):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        n_samples = len(X)
        for t in range(self.n_trees):
            # Bootstrap sample
            bootstrap_indices = [random.randrange(n_samples) for _ in range(n_samples)]
            X_boot = [X[i] for i in bootstrap_indices]
            y_boot = [y[i] for i in bootstrap_indices]
            print(f"\n=== Building Tree {t+1} ===")
            print(f"Bootstrap indices: {bootstrap_indices}")
            tree = build_tree(X_boot, y_boot, depth=0, max_depth=self.max_depth)
            self.trees.append(tree)

    def predict(self, X):
        predictions = []
        for x in X:
            pred_per_tree = [predict_tree(tree, x) for tree in self.trees]
            avg_pred = sum(pred_per_tree) / len(pred_per_tree)
            predictions.append(avg_pred)
        return predictions

# ---- Demo data ----
time_series = [10, 12, 11, 13, 14, 16, 15, 17, 18, 20]  # 10 days of values

# Create dataset with window=4 to predict next value
window = 4
X = []
y = []
for i in range(len(time_series) - window):
    X.append(time_series[i:i + window])
    y.append(time_series[i + window])

print("Input windows (X) and targets (y):")
for i in range(len(X)):
    print(f"X[{i}] = {X[i]} -> y = {y[i]}")

# Train random forest
rf = RandomForestRegressorScratch(n_trees=3, max_depth=3)
rf.fit(X[:-1], y[:-1])  # use first 5 samples for training

# Predict the next day using the last window
X_test = [X[-1]]
print("\n=== Prediction ===")
print(f"Test window: {X_test[0]}")
prediction = rf.predict(X_test)[0]
print(f"Random Forest prediction for next day: {prediction:.3f}")




Input windows (X) and targets (y):
X[0] = [10, 12, 11, 13] -> y = 14
X[1] = [12, 11, 13, 14] -> y = 16
X[2] = [11, 13, 14, 16] -> y = 15
X[3] = [13, 14, 16, 15] -> y = 17
X[4] = [14, 16, 15, 17] -> y = 18
X[5] = [16, 15, 17, 18] -> y = 20

=== Building Tree 1 ===
Bootstrap indices: [4, 4, 3, 2, 0]
Split on feature[0] <= 11.000 (depth=0, impurity=0.400)
  Split on feature[3] <= 13.000 (depth=1, impurity=0.000)
    Leaf node (depth=2): predict value = 14.000
    Leaf node (depth=2): predict value = 15.000
  Split on feature[1] <= 14.000 (depth=1, impurity=0.000)
    Leaf node (depth=2): predict value = 17.000
    Leaf node (depth=2): predict value = 18.000

=== Building Tree 2 ===
Bootstrap indices: [1, 0, 3, 2, 3]
Split on feature[2] <= 14.000 (depth=0, impurity=0.600)
  Split on feature[1] <= 11.000 (depth=1, impurity=0.333)
    Leaf node (depth=2): predict value = 16.000
    Split on feature[3] <= 13.000 (depth=2, impurity=0.000)
      Leaf node (depth=3): predict value = 14.000
     